# Genre-Extraction: demo only (seconds, no training)
**Before running** (this is what makes it work):
1. *Add Input* → **Your Work / Notebook Output** → pick the saved version of your training notebook (it contains `cnn.weights.h5`, `mel64x128.npz`, `metrics.json`), **or** attach a Dataset where you uploaded those files.
2. Accelerator can stay on *None* (CPU is enough). The GTZAN audio dataset is **not** needed: the feature cache is used.
3. Run all cells. The script finds the files by itself, so no paths to type.

In [ ]:
import os
os.makedirs("/kaggle/working/genre-demo/src/genre", exist_ok=True); os.makedirs("/kaggle/working/genre-demo/scripts", exist_ok=True)

In [ ]:
%%writefile /kaggle/working/genre-demo/src/genre/__init__.py
"""Genre-Extraction reproduction + improvements (TensorFlow 2 / Keras)."""

In [ ]:
%%writefile /kaggle/working/genre-demo/src/genre/config.py
GENRES = ["classical", "jazz", "metal", "pop"]   # same 4 classes / same label order as the original repo
SEED = 42
# colours used by the original testsoftmaxautoencoder.py for PCA plots (index == label)
GENRE_COLORS = ["navy", "green", "darkorange", "purple"]

In [ ]:
%%writefile /kaggle/working/genre-demo/src/genre/utils.py
import os
import random
import numpy as np
from sklearn.model_selection import StratifiedKFold, train_test_split

from .config import GENRES


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import tensorflow as tf
        tf.random.set_seed(seed)
    except ImportError:
        pass


def find_gtzan_root(explicit=None, genres=GENRES):
    """Return the folder that directly contains <genre>/ sub-folders full of .wav files (any nesting depth)."""
    def ok(d):
        # folders must exist AND contain .wav files (this dataset also ships Data/images_original full of PNGs)
        return all(os.path.isdir(os.path.join(d, g)) and
                   any(f.lower().endswith(".wav") for f in os.listdir(os.path.join(d, g))) for g in genres)

    if explicit and explicit != "auto":
        if ok(explicit):
            return explicit
        raise FileNotFoundError(f"{explicit} does not contain the folders {genres}")
    seen = []
    for base in ("/kaggle/input", "./genres", "./data", "."):
        if not os.path.isdir(base):
            continue
        for dirpath, dirnames, _ in os.walk(base):
            if ok(dirpath):                      # check BEFORE anything else; no depth limit
                return dirpath
            if len(seen) < 25 and dirpath.count(os.sep) <= 6:
                seen.append(dirpath)
            if "outputs" in dirnames:            # never descend into our own results
                dirnames.remove("outputs")
    raise FileNotFoundError(
        f"No folder containing {genres} found. Folders seen (first 25):\n  " + "\n  ".join(seen) +
        "\nPass --data-root <folder that contains classical/, jazz/, metal/, pop/>")


def list_wavs(root, genre):
    d = os.path.join(root, genre)
    return [os.path.join(d, f) for f in sorted(os.listdir(d)) if f.lower().endswith(".wav")]


def song_split(song_genre, val_frac=0.15, test_frac=0.15, seed=42, n_folds=0, fold=0):
    """Stratified split on SONGS (never on segments). Returns (train, val, test) song indices."""
    idx = np.arange(len(song_genre))
    song_genre = np.asarray(song_genre)
    if n_folds and n_folds > 1:
        skf = StratifiedKFold(n_folds, shuffle=True, random_state=seed)
        rest, test = list(skf.split(idx, song_genre))[fold]
        tr, va = train_test_split(rest, test_size=val_frac, stratify=song_genre[rest], random_state=seed)
    else:
        rest, test = train_test_split(idx, test_size=test_frac, stratify=song_genre, random_state=seed)
        tr, va = train_test_split(rest, test_size=val_frac / (1 - test_frac),
                                  stratify=song_genre[rest], random_state=seed)
    return np.sort(tr), np.sort(va), np.sort(test)


def make_splits(X, y, song, mode="song", seed=42, n_folds=0, fold=0):
    """mode='random_segment' -> what the original repo did (leaky); mode='song' -> leak-free."""
    X, y, song = np.asarray(X), np.asarray(y), np.asarray(song)
    n = len(X)
    if mode == "random_segment":
        perm = np.random.RandomState(seed).permutation(n)
        a, b = int(0.75 * n), int(0.875 * n)          # 6000 / 1000 / 1000 for n = 8000
        parts = {"train": perm[:a], "val": perm[a:b], "test": perm[b:]}
    elif mode == "song":
        n_songs = song.max() + 1
        sg = np.zeros(n_songs, dtype=int)
        sg[song] = y
        tr, va, te = song_split(sg, seed=seed, n_folds=n_folds, fold=fold)
        parts = {k: np.where(np.isin(song, s))[0] for k, s in zip(("train", "val", "test"), (tr, va, te))}
    else:
        raise ValueError(mode)
    return {k: (X[i], y[i], song[i]) for k, i in parts.items()}


def leakage(splits):
    """Fraction of test segments whose parent song also appears in the training set."""
    tr_songs = np.unique(splits["train"][2])
    return float(np.isin(splits["test"][2], tr_songs).mean())

In [ ]:
%%writefile /kaggle/working/genre-demo/src/genre/data_mel.py
"""Improved front-end: log-mel spectrogram segments (3 s, 50 % overlap) instead of 551 Hz average-pooled waveform."""
import os
import warnings
import numpy as np

from .config import GENRES
from .utils import list_wavs, make_splits, leakage

SR, SEG_SEC, HOP_SEC = 22050, 3.0, 1.5
N_FFT, HOP, N_MELS, N_FRAMES = 1024, 512, 64, 128


def song_to_segments(path, librosa):
    y, _ = librosa.load(path, sr=SR, mono=True, duration=30.0)
    seg, step = int(SR * SEG_SEC), int(SR * HOP_SEC)
    out = []
    for start in range(0, max(len(y) - seg, 0) + 1, step):
        chunk = y[start:start + seg]
        if len(chunk) < seg:
            chunk = np.pad(chunk, (0, seg - len(chunk)))
        m = librosa.feature.melspectrogram(y=chunk, sr=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS, fmin=20)
        m = librosa.power_to_db(m, ref=1.0, top_db=80.0)      # absolute level kept (loudness is a genre cue)
        out.append(m[:, :N_FRAMES].astype(np.float32))        # (64, 130) -> (64, 128)
    return out


def load_mel_dataset(root, genres=GENRES, cache=None):
    if cache and os.path.exists(cache):
        d = np.load(cache, allow_pickle=True)
        if d["X"].size:                                   # ignore empty/corrupt caches
            return d["X"], d["y"], d["song"], list(d["names"])
    import librosa
    X, y, song, names = [], [], [], []
    for gi, g in enumerate(genres):
        for path in list_wavs(root, g):
            try:
                segs = song_to_segments(path, librosa)
            except Exception as e:
                warnings.warn(f"skipping unreadable file {path}: {e}")
                continue
            sid = len(names)
            names.append(os.path.basename(path))
            for s in segs:
                X.append(s)
                y.append(gi)
                song.append(sid)
    if not names:
        raise RuntimeError(f"No .wav files could be read under {root}. Is this the genres_original folder?")
    X = np.asarray(X, dtype=np.float32)
    y, song = np.asarray(y, dtype=np.int32), np.asarray(song, dtype=np.int32)
    if cache:
        os.makedirs(os.path.dirname(cache) or ".", exist_ok=True)
        np.savez(cache, X=X, y=y, song=song, names=np.asarray(names))
    return X, y, song, names


def prepare_mel(root, cache=None, split="song", seed=42, n_folds=0, fold=0):
    X, y, song, names = load_mel_dataset(root, cache=cache)
    splits = make_splits(X, y, song, mode=split, seed=seed, n_folds=n_folds, fold=fold)
    mu, sd = splits["train"][0].mean(), splits["train"][0].std() + 1e-6      # train-only statistics
    splits = {k: (((a - mu) / sd)[..., None].astype(np.float32), yy, ss) for k, (a, yy, ss) in splits.items()}
    print(f"[data] {len(names)} songs -> {len(X)} segments | split={split} | "
          f"test-segments whose song is also in train: {leakage(splits):.1%}")
    return splits

In [ ]:
%%writefile /kaggle/working/genre-demo/src/genre/evaluate.py
import os
import json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.decomposition import PCA

from .config import GENRES, GENRE_COLORS


def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def song_level(probs, y, song):
    """Average segment probabilities per song -> one prediction per song."""
    ids = np.unique(song)
    P = np.stack([probs[song == i].mean(axis=0) for i in ids])
    T = np.array([y[song == i][0] for i in ids])
    return T, P.argmax(1)


def summarize(logits, y, song, genres=GENRES, name=""):
    probs = softmax(logits)
    pred = probs.argmax(1)
    st, sp = song_level(probs, y, song)
    rep = classification_report(y, pred, target_names=genres, output_dict=True, zero_division=0)
    res = {"segment_acc": float(accuracy_score(y, pred)),
           "segment_macro_f1": float(f1_score(y, pred, average="macro")),
           "song_acc": float(accuracy_score(st, sp)),
           "n_segments": int(len(y)), "n_songs": int(len(st)),
           "per_class": {g: {k: float(rep[g][k]) for k in ("precision", "recall", "f1-score")} for g in genres}}
    print(f"\n== {name} ==  segment acc {res['segment_acc']:.3f} | macro-F1 {res['segment_macro_f1']:.3f} | "
          f"song acc {res['song_acc']:.3f} ({res['n_songs']} songs)")
    print(classification_report(y, pred, target_names=genres, digits=3, zero_division=0))
    return res, pred


def plot_confusion(y, pred, path, title="Normalized confusion matrix", genres=GENRES):
    cm = confusion_matrix(y, pred, labels=range(len(genres))).astype(float)
    cm = cm / cm.sum(axis=1, keepdims=True)
    fig, ax = plt.subplots(figsize=(5, 4.5))
    im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(genres)))
    ax.set_xticklabels(genres, rotation=45)
    ax.set_yticks(range(len(genres)))
    ax.set_yticklabels(genres)
    for i in range(len(genres)):
        for j in range(len(genres)):
            ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", color="white" if cm[i, j] > .5 else "black")
    ax.set_xlabel("Predicted label"); ax.set_ylabel("True label"); ax.set_title(title)
    fig.colorbar(im); fig.tight_layout(); fig.savefig(path, dpi=130); plt.close(fig)


def plot_pca(Z, y, title, path, genres=GENRES):
    Z2 = PCA(n_components=2).fit_transform(Z.reshape(len(Z), -1))
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    for k in reversed(range(len(genres))):                      # same draw order as original plots
        m = y == k
        ax.scatter(Z2[m, 0], Z2[m, 1], c=GENRE_COLORS[k], alpha=.5, edgecolor="k", s=18, label=genres[k].capitalize())
    ax.set_title(title); ax.legend(loc="upper right")
    fig.tight_layout(); fig.savefig(path, dpi=130); plt.close(fig)


def plot_history(hist, path, title="Training curves"):
    keys = [k for k in hist if hist[k]]
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
    for k in keys:
        (axes[0] if "loss" in k else axes[1]).plot(hist[k], label=k)
    axes[0].set_title("loss"); axes[1].set_title("accuracy")
    for a in axes:
        a.legend(fontsize=7); a.set_xlabel("epoch")
    fig.suptitle(title); fig.tight_layout(); fig.savefig(path, dpi=130); plt.close(fig)


def save_json(obj, path):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

In [ ]:
%%writefile /kaggle/working/genre-demo/src/genre/models_improved.py
"""Better front-end + back-end:  log-mel (64 x 128 x 1) -> small VGG-style CNN.

build_cnn        : pure classifier, 64-d 'embedding' layer before the softmax.
build_conv_dsae  : the paper's idea (autoencoder + classifier on the SAME 64-d code) but convolutional.
                   Trained as a *denoising* AE when SpecAugment is on (input masked, target clean).
"""
from tensorflow.keras import layers, Model, optimizers, losses


def _conv_block(x, f):
    x = layers.Conv2D(f, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(f, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    return layers.Activation("relu")(x)


def build_cnn(input_shape=(64, 128, 1), n_classes=4, base=32, latent=64, dropout=0.3, lr=1e-3, **_):
    inp = layers.Input(input_shape, name="x")
    x = inp
    for f in (base, 2 * base, 4 * base, 4 * base):
        x = _conv_block(x, f)
        x = layers.MaxPooling2D(2)(x)
        x = layers.Dropout(dropout / 2)(x)
    x = layers.GlobalAveragePooling2D()(x)
    z = layers.Dense(latent, activation="relu", name="embedding")(x)
    z = layers.Dropout(dropout)(z)
    logits = layers.Dense(n_classes, name="logits")(z)
    m = Model(inp, logits, name="mel_cnn")
    m.compile(optimizers.Adam(lr), loss=losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=["sparse_categorical_accuracy"])
    return m


def build_conv_dsae(input_shape=(64, 128, 1), n_classes=4, base=32, latent=64, dropout=0.3, lr=1e-3,
                    recon_weight=1.0, cls_weight=1.0, **_):
    inp = layers.Input(input_shape, name="x")
    x = inp
    for f in (base, 2 * base, 4 * base, 4 * base):          # 64x128 -> 4x8
        x = _conv_block(x, f)
        x = layers.MaxPooling2D(2)(x)
    h, w, c = input_shape[0] // 16, input_shape[1] // 16, 4 * base
    x = layers.Flatten()(x)
    x = layers.Dropout(dropout)(x)
    z = layers.Dense(latent, activation="relu", name="embedding")(x)
    # classifier head: same shape as the paper (64 -> 32 -> 16 -> classes, tanh)
    k = layers.Dense(32, activation="tanh")(z)
    k = layers.Dense(16, activation="tanh")(k)
    logits = layers.Dense(n_classes, name="logits")(k)
    # decoder
    d = layers.Dense(h * w * c, activation="relu")(z)
    d = layers.Reshape((h, w, c))(d)
    for f in (4 * base, 2 * base, base, max(base // 2, 8)):  # 4x8 -> 64x128
        d = layers.UpSampling2D(2)(d)
        d = _conv_block(d, f)
    recon = layers.Conv2D(1, 3, padding="same", name="recon")(d)
    m = Model(inp, {"recon": recon, "logits": logits}, name="conv_deep_softmax_autoencoder")
    m.compile(optimizers.Adam(lr),
              loss={"recon": "mse", "logits": losses.SparseCategoricalCrossentropy(from_logits=True)},
              loss_weights={"recon": recon_weight, "logits": cls_weight},
              metrics={"logits": ["sparse_categorical_accuracy"]})
    return m

In [ ]:
%%writefile /kaggle/working/genre-demo/scripts/04_demo.py
"""Live demo of the trained mel-CNN (no training, takes seconds once features are cached).

  python scripts/04_demo.py --weights outputs/cnn/cnn.weights.h5 --n 10
  python scripts/04_demo.py --weights outputs/cnn/cnn.weights.h5 --audio my_song.wav other.wav

What it does
 1. Re-creates the exact song-level split (seed 42) and the train-only standardisation.
 2. Loads the weights and RE-COMPUTES accuracy on the 60 unseen test songs; compares with the saved metrics.json.
 3. Shows N random test songs: true genre, predicted genre, confidence.
 4. For every --audio file: predicts its genre and saves a picture (spectrogram + probabilities).
"""
import argparse, json, os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "src"))
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from genre.config import GENRES
from genre.data_mel import load_mel_dataset, song_to_segments
from genre.evaluate import softmax
from genre.utils import find_gtzan_root, make_splits


def locate(path, filename):
    """Return `path` if it exists, otherwise search /kaggle/input, /kaggle/working and . for `filename`."""
    if path and os.path.exists(path):
        return path
    for base in ("/kaggle/input", "/kaggle/working", "."):
        if not os.path.isdir(base):
            continue
        for dirpath, _, files in os.walk(base):
            if filename in files:
                found = os.path.join(dirpath, filename)
                print(f"[locate] {filename} -> {found}")
                return found
    raise FileNotFoundError(f"{filename} not found. Attach the notebook output / dataset that contains it "
                            f"(Add Input), or train it first (scripts/02_train_improved.py --model cnn).")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--weights", default="outputs/cnn/cnn.weights.h5")
    ap.add_argument("--data-root", default="auto")
    ap.add_argument("--cache", default="outputs/cache/mel64x128.npz")
    ap.add_argument("--audio", nargs="*", default=[])
    ap.add_argument("--n", type=int, default=10)
    ap.add_argument("--out", default="outputs/demo")
    ap.add_argument("--seed", type=int, default=42)
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)

    # 1) same data, same split, same normalisation as training
    a.weights = locate(a.weights, os.path.basename(a.weights))
    cache = a.cache
    if not os.path.exists(cache):
        try:
            cache = locate(cache, os.path.basename(cache))
        except FileNotFoundError:
            pass                                   # fall back to rebuilding features from the audio
    root = None if os.path.exists(cache) else find_gtzan_root(a.data_root)   # audio dataset only needed without a cache
    X, y, song, names = load_mel_dataset(root, cache=cache)
    sp = make_splits(X, y, song, mode="song", seed=a.seed)
    mu, sd = sp["train"][0].mean(), sp["train"][0].std() + 1e-6
    Xte, yte, ste = sp["test"]
    Xte = ((Xte - mu) / sd)[..., None].astype(np.float32)

    # 2) model
    from genre.models_improved import build_cnn
    model = build_cnn((64, 128, 1))
    model.load_weights(a.weights)
    print(f"loaded {a.weights}\n")

    probs = softmax(model.predict(Xte, batch_size=256, verbose=0))
    seg_acc = float((probs.argmax(1) == yte).mean())
    ids = np.unique(ste)
    P = np.stack([probs[ste == i].mean(0) for i in ids])
    T = np.array([yte[ste == i][0] for i in ids])
    song_acc = float((P.argmax(1) == T).mean())
    print(f"Unseen TEST songs: {len(ids)} | segment accuracy {seg_acc:.3f} | song accuracy {song_acc:.3f}")
    mj = os.path.join(os.path.dirname(a.weights), "metrics.json")
    if os.path.exists(mj):
        saved = json.load(open(mj))["test"]
        print(f"saved in metrics.json:             segment accuracy {saved['segment_acc']:.3f} | song accuracy {saved['song_acc']:.3f}")

    # 3) a few songs, one by one
    rng = np.random.RandomState(0)
    pick = rng.choice(len(ids), size=min(a.n, len(ids)), replace=False)
    print(f"\n{'song file':<22}{'true':<11}{'predicted':<11}{'confidence':<11}")
    for k in pick:
        p = P[k]
        mark = "OK " if p.argmax() == T[k] else "WRONG"
        print(f"{names[ids[k]]:<22}{GENRES[T[k]]:<11}{GENRES[p.argmax()]:<11}{p.max():<11.1%}{mark}")

    # 4) your own audio
    for path in a.audio:
        segs = np.stack(song_to_segments(path, __import__("librosa")))
        x = ((segs - mu) / sd)[..., None].astype(np.float32)
        p = softmax(model.predict(x, verbose=0)).mean(0)
        print(f"\n{os.path.basename(path)}: {len(segs)} windows -> {GENRES[p.argmax()]} ({p.max():.1%})  | " +
              ", ".join(f"{g} {v:.1%}" for g, v in zip(GENRES, p)))
        fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
        ax[0].imshow(segs[0], origin="lower", aspect="auto"); ax[0].set_title("log-mel spectrogram (first 3 s)")
        ax[0].set_xlabel("time frames"); ax[0].set_ylabel("mel band")
        ax[1].bar(GENRES, p); ax[1].set_ylim(0, 1); ax[1].set_title(f"prediction: {GENRES[p.argmax()]} ({p.max():.0%})")
        fig.tight_layout()
        fp = os.path.join(a.out, "demo_" + os.path.splitext(os.path.basename(path))[0] + ".png")
        fig.savefig(fp, dpi=120); plt.close(fig)
        print("saved", fp)


if __name__ == "__main__":
    main()

## Files found (if this prints nothing, attach the training notebook's output as an Input)

In [ ]:
!find /kaggle/input /kaggle/working -name "cnn.weights.h5" -o -name "mel64x128.npz" -o -name "metrics.json" 2>/dev/null | head -20

## 1. Recompute accuracy on the 60 unseen test songs and list songs (`--n 60` lists all of them)

In [ ]:
!cd /kaggle/working/genre-demo && python scripts/04_demo.py --n 12

## 2. Your own audio (edit the path: Add Input / Upload a .wav, then use the path shown in the sidebar)

In [ ]:
!cd /kaggle/working/genre-demo && python scripts/04_demo.py --n 0 --audio /kaggle/input/YOUR_DATASET/my_song.wav

In [ ]:
import glob
from IPython.display import Image, display
for f in sorted(glob.glob('/kaggle/working/genre-demo/outputs/demo/*.png')): display(Image(f))